In [ ]:
# === Cell 1: Mount Drive, install nnU-Net v2, set env vars ===

from google.colab import drive
drive.mount('/content/drive')

!pip install nnunetv2

import os

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
os.makedirs(os.path.join(WORKSPACE, 'nnUNet_raw'), exist_ok=True)
os.makedirs(os.path.join(WORKSPACE, 'nnUNet_preprocessed'), exist_ok=True)

for run_id in range(3):
    os.makedirs(os.path.join(WORKSPACE, f'nnUNet_results_run{run_id}'), exist_ok=True)

os.environ['nnUNet_raw'] = os.path.join(WORKSPACE, 'nnUNet_raw')
os.environ['nnUNet_preprocessed'] = os.path.join(WORKSPACE, 'nnUNet_preprocessed')
# nnUNet_results will be set per-run in Cell 6; default to run0 for now
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run0')

print('nnUNet_raw:', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed:', os.environ['nnUNet_preprocessed'])
print('nnUNet_results (default):', os.environ['nnUNet_results'])

In [ ]:
# === Cell 2: Convert dataset to nnU-Net format ===

import os, shutil, json
import numpy as np
from PIL import Image

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RAW = os.environ['nnUNet_raw']
DATASET = os.path.join(RAW, 'Dataset501_VFSS')

imagesTr = os.path.join(DATASET, 'imagesTr')
labelsTr = os.path.join(DATASET, 'labelsTr')
imagesTs = os.path.join(DATASET, 'imagesTs')

# Clear any existing contents to avoid stale files from previous runs
for d in [imagesTr, labelsTr, imagesTs]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

# Copy train + val images; remap masks to {0,1}
num_training = 0
for split in ['train', 'val']:
    img_dir = os.path.join(BASE, split, 'images')
    msk_dir = os.path.join(BASE, split, 'masks')
    for fname in sorted(os.listdir(img_dir)):
        if not fname.endswith('.png'):
            continue
        stem = fname.replace('.png', '')

        # Image: load, convert to single-channel grayscale, save with _0000 suffix
        Image.open(os.path.join(img_dir, fname)).convert('L').save(
            os.path.join(imagesTr, f'{stem}_0000.png'),
        )

        # Mask: load, remap to {0,1}, save as uint8 PNG
        mask = np.array(Image.open(os.path.join(msk_dir, fname)).convert('L'))
        mask_bin = (mask > 0).astype(np.uint8)  # {0,255} -> {0,1}
        Image.fromarray(mask_bin).save(os.path.join(labelsTr, f'{stem}.png'))

        num_training += 1

print(f'Copied {num_training} training+val cases to imagesTr/labelsTr')

# Verify ALL converted labels contain only {0,1}
for label_fname in sorted(os.listdir(labelsTr)):
    label_arr = np.array(Image.open(os.path.join(labelsTr, label_fname)))
    unique_vals = set(np.unique(label_arr))
    assert unique_vals.issubset({0, 1}), (
        f'Label {label_fname} has unexpected values: {unique_vals}'
    )
print(f'Verified: all {num_training} labels contain only values in {{0, 1}}')

# Verify ALL training images are single-channel grayscale
for img_fname in sorted(os.listdir(imagesTr)):
    mode = Image.open(os.path.join(imagesTr, img_fname)).mode
    assert mode == 'L', f'imagesTr/{img_fname} is {mode}, expected L (grayscale)'
print(f'Verified: all {num_training} imagesTr PNGs are single-channel (L)')

# Copy test images into imagesTs
num_test = 0
test_img_dir = os.path.join(BASE, 'test', 'images')
for fname in sorted(os.listdir(test_img_dir)):
    if not fname.endswith('.png'):
        continue
    stem = fname.replace('.png', '')
    Image.open(os.path.join(test_img_dir, fname)).convert('L').save(
        os.path.join(imagesTs, f'{stem}_0000.png'),
    )
    num_test += 1

print(f'Copied {num_test} test cases to imagesTs')

# Verify ALL test images are single-channel grayscale
for img_fname in sorted(os.listdir(imagesTs)):
    mode = Image.open(os.path.join(imagesTs, img_fname)).mode
    assert mode == 'L', f'imagesTs/{img_fname} is {mode}, expected L (grayscale)'
print(f'Verified: all {num_test} imagesTs PNGs are single-channel (L)')

# Create dataset.json
dataset_json = {
    'channel_names': {'0': 'Xray'},
    'labels': {'background': 0, 'vertebra': 1},
    'numTraining': num_training,
    'file_ending': '.png',
}

with open(os.path.join(DATASET, 'dataset.json'), 'w') as f:
    json.dump(dataset_json, f, indent=2)

print(f'dataset.json saved with numTraining={num_training}')

In [ ]:
# === Cell 3: Build custom split (train/val from original split) ===

import os, json

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'

train_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'train', 'images'))
    if f.endswith('.png')
])

val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])

splits_final = [{
    'train': train_stems,
    'val': val_stems,
}]

print(f'Train stems: {len(train_stems)}')
print(f'Val stems:   {len(val_stems)}')
print(f'Total:       {len(train_stems) + len(val_stems)}')
print(f'First train: {train_stems[:3]}')
print(f'First val:   {val_stems[:3]}')

# Save temporarily; will copy to preprocessed folder after Cell 4
_SPLITS_TMP = '/content/splits_final.json'
with open(_SPLITS_TMP, 'w') as f:
    json.dump(splits_final, f, indent=2)
print(f'Temporary splits saved to {_SPLITS_TMP}')

In [ ]:
# === Cell 4: Planning and preprocessing ===

!nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity -c 2d

In [ ]:
# === Cell 5: Write custom splits_final.json to preprocessed folder ===

import os, json, shutil

preprocessed_dir = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS'
)
assert os.path.isdir(preprocessed_dir), (
    f'Preprocessed folder not found: {preprocessed_dir}. Run Cell 4 first.'
)

splits_dst = os.path.join(preprocessed_dir, 'splits_final.json')
shutil.copy2('/content/splits_final.json', splits_dst)

# Verify
with open(splits_dst) as f:
    splits = json.load(f)

assert len(splits) == 1, f'Expected 1 fold, got {len(splits)}'
print(f'splits_final.json written to {splits_dst}')
print(f'  train: {len(splits[0]["train"])} cases')
print(f'  val:   {len(splits[0]["val"])} cases')

# Cross-check with dataset.json
ds_path = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'dataset.json')
with open(ds_path) as f:
    ds = json.load(f)
n_split = len(splits[0]['train']) + len(splits[0]['val'])
assert ds['numTraining'] == n_split, (
    f'Mismatch: numTraining={ds["numTraining"]} vs split total={n_split}'
)
print(f'Verified: numTraining ({ds["numTraining"]}) == train+val ({n_split})')

In [ ]:
# === Cell 6a: Train run_0 ===
# Training runs for 1000 epochs (~3-6 hours per run on Colab).
# nnU-Net has no seed parameter; we run 3 times to estimate run-to-run variability
# arising from non-deterministic data augmentation and weight initialization.

import os, gc, torch, time

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_run0')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume if Colab disconnects: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0

fold_dir = os.path.join(
    results_dir,
    'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'
)
print(f'
Run 0 done in {elapsed/3600:.1f}h')
print(f'Checkpoints: {fold_dir}')
if os.path.isdir(fold_dir):
    print('Contents:', os.listdir(fold_dir))

In [ ]:
# === Cell 6b: Train run_1 ===
# Training runs for 1000 epochs (~3-6 hours per run on Colab).
# nnU-Net has no seed parameter; we run 3 times to estimate run-to-run variability
# arising from non-deterministic data augmentation and weight initialization.

import os, gc, torch, time

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_run1')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume if Colab disconnects: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0

fold_dir = os.path.join(
    results_dir,
    'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'
)
print(f'
Run 1 done in {elapsed/3600:.1f}h')
print(f'Checkpoints: {fold_dir}')
if os.path.isdir(fold_dir):
    print('Contents:', os.listdir(fold_dir))

In [ ]:
# === Cell 6c: Train run_2 ===
# Training runs for 1000 epochs (~3-6 hours per run on Colab).
# nnU-Net has no seed parameter; we run 3 times to estimate run-to-run variability
# arising from non-deterministic data augmentation and weight initialization.

import os, gc, torch, time

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_run2')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume if Colab disconnects: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0

fold_dir = os.path.join(
    results_dir,
    'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'
)
print(f'
Run 2 done in {elapsed/3600:.1f}h')
print(f'Checkpoints: {fold_dir}')
if os.path.isdir(fold_dir):
    print('Contents:', os.listdir(fold_dir))

In [ ]:
# === Cell 7a: Predict from run_0 ===

import os, shutil

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')

os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run0')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_run0')

# Clear any existing predictions to avoid stale files
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict     -i {imagesTs}     -o {pred_dir}     -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Run 0: {len(preds)} predictions saved to {pred_dir}')

In [ ]:
# === Cell 7b: Predict from run_1 ===

import os, shutil

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')

os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run1')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_run1')

# Clear any existing predictions to avoid stale files
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict     -i {imagesTs}     -o {pred_dir}     -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Run 1: {len(preds)} predictions saved to {pred_dir}')

In [ ]:
# === Cell 7c: Predict from run_2 ===

import os, shutil

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')

os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run2')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_run2')

# Clear any existing predictions to avoid stale files
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict     -i {imagesTs}     -o {pred_dir}     -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Run 2: {len(preds)} predictions saved to {pred_dir}')

In [ ]:
# === Cell 8: Evaluate all 3 runs, compute mean +/- std, save results ===

import os, json, csv, re, shutil, time
import numpy as np
from PIL import Image

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
WORKSPACE = os.path.join(BASE, 'nnunet_workspace')
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')
GT_DIR = os.path.join(BASE, 'test', 'masks')
os.makedirs(RESULTS_BASE, exist_ok=True)

gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    """Strip .png and any trailing _NNNN channel suffix (e.g. _0000)."""
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


all_run_results = []
all_run_per_image = []

for run_id in range(3):
    pred_dir = os.path.join(RESULTS_BASE, f'predictions_run{run_id}')
    pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
    pred_by_stem = {
        normalize_pred_stem(f): os.path.join(pred_dir, f)
        for f in pred_files
    }

    matched_stems = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
    assert len(matched_stems) == len(gt_files), (
        f'Run {run_id}: expected {len(gt_files)} matches, got {len(matched_stems)}. '
        f'Unmatched GT: {set(gt_by_stem.keys()) - set(pred_by_stem.keys())}'
    )

    per_image = []
    for stem in matched_stems:
        gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
        pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))

        if pred.shape != gt.shape:
            pred = np.array(
                Image.fromarray(pred).resize(
                    (gt.shape[1], gt.shape[0]), Image.NEAREST
                )
            )

        gt_bin = (gt > 0).astype(np.float32)
        pred_bin = (pred > 0).astype(np.float32)
        dice, iou = compute_dice_iou(pred_bin, gt_bin)
        per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

    mean_f1 = np.mean([r['f1'] for r in per_image])
    mean_iou = np.mean([r['iou'] for r in per_image])

    all_run_results.append({'run_id': run_id, 'mean_f1': mean_f1, 'mean_iou': mean_iou})
    all_run_per_image.append(per_image)

    # Save per-run CSV
    csv_path = os.path.join(RESULTS_BASE, f'per_image_metrics_run{run_id}.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
        writer.writeheader()
        writer.writerows(per_image)

    print(f'Run {run_id}: F1={mean_f1:.4f}, IoU={mean_iou:.4f} ({len(per_image)} images)')

# Aggregate across 3 runs
f1_values = [r['mean_f1'] for r in all_run_results]
iou_values = [r['mean_iou'] for r in all_run_results]

agg_f1_mean = float(np.mean(f1_values))
agg_f1_std = float(np.std(f1_values))
agg_iou_mean = float(np.mean(iou_values))
agg_iou_std = float(np.std(iou_values))

print(f'\n{"="*50}')
print(f'Aggregated (3 runs):')
print(f'  F1 (Dice): {agg_f1_mean:.4f} +/- {agg_f1_std:.4f}')
print(f'  IoU:       {agg_iou_mean:.4f} +/- {agg_iou_std:.4f}')
print(f'{"="*50}')

# Save aggregated metrics.json
metrics = {
    'mean_f1': agg_f1_mean,
    'std_f1': agg_f1_std,
    'mean_iou': agg_iou_mean,
    'std_iou': agg_iou_std,
    'per_run': [
        {'run_id': r['run_id'], 'mean_f1': float(r['mean_f1']), 'mean_iou': float(r['mean_iou'])}
        for r in all_run_results
    ],
}
with open(os.path.join(RESULTS_BASE, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print(f'Saved: metrics.json')

# Load nnUNet plans for run_report
plans_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'nnUNetPlans.json'
)
plans_summary = {}
if os.path.isfile(plans_path):
    with open(plans_path) as f:
        plans = json.load(f)
    cfg_2d = plans.get('configurations', {}).get('2d', {})
    plans_summary = {
        'patch_size': cfg_2d.get('patch_size'),
        'batch_size': cfg_2d.get('batch_size'),
        'architecture': cfg_2d.get('architecture'),
        'spacing': cfg_2d.get('spacing'),
    }
    shutil.copy2(plans_path, os.path.join(RESULTS_BASE, 'nnUNetPlans.json'))
    print('Copied nnUNetPlans.json to results folder')

# Build run_report.json
run_report = {
    'experiment': 'nnunet_baseline',
    'dataset_id': 501,
    'configuration': '2d',
    'num_runs': 3,
    'epochs_per_run': 1000,
    'mean_f1': agg_f1_mean,
    'std_f1': agg_f1_std,
    'mean_iou': agg_iou_mean,
    'std_iou': agg_iou_std,
    'per_run_results': [
        {'run_id': r['run_id'], 'mean_f1': float(r['mean_f1']), 'mean_iou': float(r['mean_iou'])}
        for r in all_run_results
    ],
    'num_test_images': len(gt_files),
    'nnunet_config_summary': plans_summary,
    'note': '3 independent runs to estimate run-to-run variability. '
            'nnU-Net has no seed parameter; variability may arise from '
            'non-deterministic data augmentation and weight initialization.',
    'paths': {
        'predictions_run0': os.path.join(RESULTS_BASE, 'predictions_run0'),
        'predictions_run1': os.path.join(RESULTS_BASE, 'predictions_run1'),
        'predictions_run2': os.path.join(RESULTS_BASE, 'predictions_run2'),
        'checkpoints_run0': os.path.join(WORKSPACE, 'nnUNet_results_run0',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'checkpoints_run1': os.path.join(WORKSPACE, 'nnUNet_results_run1',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'checkpoints_run2': os.path.join(WORKSPACE, 'nnUNet_results_run2',
            'Dataset501_VFSS', 'nnUNetTrainer__nnUNetPlans__2d', 'fold_0'),
        'nnUNetPlans': plans_path,
    },
    'timestamp_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
}

with open(os.path.join(RESULTS_BASE, 'run_report.json'), 'w') as f:
    json.dump(run_report, f, indent=2)
print(f'Saved: run_report.json')

# --- Final verification ---
print('\n--- Verification ---')

splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
if os.path.isfile(splits_path):
    with open(splits_path) as f:
        splits = json.load(f)
    assert len(splits) == 1, f'splits_final.json has {len(splits)} folds, expected 1'
    print('OK: splits_final.json has 1 fold')

    ds_path = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'dataset.json')
    with open(ds_path) as f:
        ds = json.load(f)
    n_split = len(splits[0]['train']) + len(splits[0]['val'])
    assert ds['numTraining'] == n_split
    print(f'OK: numTraining ({ds["numTraining"]}) == train+val ({n_split})')

for run_id in range(3):
    assert len(all_run_per_image[run_id]) == len(gt_files), (
        f'Run {run_id}: expected {len(gt_files)} results, got {len(all_run_per_image[run_id])}'
    )
print(f'OK: All 3 runs matched {len(gt_files)} test predictions to GT')

print(f'\nDone. F1={agg_f1_mean:.4f} +/- {agg_f1_std:.4f}, IoU={agg_iou_mean:.4f} +/- {agg_iou_std:.4f}')

In [ ]:
# === Cell A: Train + predict + evaluate with 25% labels ===

import os, gc, json, csv, re, shutil, time
import torch
import numpy as np
from PIL import Image

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')

# --- Read train stems from label fraction file ---
with open('/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_25/stems.txt') as f:
    train_stems = sorted([line.strip() for line in f if line.strip()])
print(f'Train stems (25%): {len(train_stems)}')

# --- Build val stems (always the same 44) ---
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'
assert not set(train_stems) & set(val_stems), 'Train/val overlap detected!'
print(f'Val stems: {len(val_stems)}')

# --- Overwrite splits_final.json ---
splits_final = [{"train": train_stems, "val": val_stems}]
splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump(splits_final, f, indent=2)
print(f'splits_final.json overwritten: {len(train_stems)} train + {len(val_stems)} val')

# --- Train ---
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_25pct')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0
print(f'Training done in {elapsed/3600:.1f}h')

# --- Predict ---
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_25pct')
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Predictions: {len(preds)} files in {pred_dir}')

# --- Evaluate ---
GT_DIR = os.path.join(BASE, 'test', 'masks')
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}

matched_stems = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
assert len(matched_stems) == len(gt_files), (
    f'Expected {len(gt_files)} matches, got {len(matched_stems)}'
)

per_image = []
for stem in matched_stems:
    gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
    pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))
    if pred.shape != gt.shape:
        pred = np.array(
            Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST)
        )
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

mean_f1 = float(np.mean([r['f1'] for r in per_image]))
mean_iou = float(np.mean([r['iou'] for r in per_image]))

# Save per-image CSV
csv_path = os.path.join(RESULTS_BASE, 'per_image_metrics_25pct.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
    writer.writeheader()
    writer.writerows(per_image)

# Save metrics JSON
metrics = {
    'fraction': 25,
    'num_train': len(train_stems),
    'num_val': len(val_stems),
    'mean_f1': mean_f1,
    'mean_iou': mean_iou,
    'num_test_images': len(per_image),
}
with open(os.path.join(RESULTS_BASE, 'metrics_25pct.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n25% labels ({len(train_stems)} train): F1={mean_f1:.4f}, IoU={mean_iou:.4f}')

In [ ]:
# === Cell B: Train + predict + evaluate with 50% labels ===

import os, gc, json, csv, re, shutil, time
import torch
import numpy as np
from PIL import Image

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')

# --- Read train stems from label fraction file ---
with open('/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_50/stems.txt') as f:
    train_stems = sorted([line.strip() for line in f if line.strip()])
print(f'Train stems (50%): {len(train_stems)}')

# --- Build val stems (always the same 44) ---
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'
assert not set(train_stems) & set(val_stems), 'Train/val overlap detected!'
print(f'Val stems: {len(val_stems)}')

# --- Overwrite splits_final.json ---
splits_final = [{"train": train_stems, "val": val_stems}]
splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump(splits_final, f, indent=2)
print(f'splits_final.json overwritten: {len(train_stems)} train + {len(val_stems)} val')

# --- Train ---
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_50pct')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0
print(f'Training done in {elapsed/3600:.1f}h')

# --- Predict ---
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_50pct')
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Predictions: {len(preds)} files in {pred_dir}')

# --- Evaluate ---
GT_DIR = os.path.join(BASE, 'test', 'masks')
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}

matched_stems = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
assert len(matched_stems) == len(gt_files), (
    f'Expected {len(gt_files)} matches, got {len(matched_stems)}'
)

per_image = []
for stem in matched_stems:
    gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
    pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))
    if pred.shape != gt.shape:
        pred = np.array(
            Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST)
        )
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

mean_f1 = float(np.mean([r['f1'] for r in per_image]))
mean_iou = float(np.mean([r['iou'] for r in per_image]))

# Save per-image CSV
csv_path = os.path.join(RESULTS_BASE, 'per_image_metrics_50pct.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
    writer.writeheader()
    writer.writerows(per_image)

# Save metrics JSON
metrics = {
    'fraction': 50,
    'num_train': len(train_stems),
    'num_val': len(val_stems),
    'mean_f1': mean_f1,
    'mean_iou': mean_iou,
    'num_test_images': len(per_image),
}
with open(os.path.join(RESULTS_BASE, 'metrics_50pct.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n50% labels ({len(train_stems)} train): F1={mean_f1:.4f}, IoU={mean_iou:.4f}')

In [ ]:
# === Cell C: Train + predict + evaluate with 75% labels ===

import os, gc, json, csv, re, shutil, time
import torch
import numpy as np
from PIL import Image

gc.collect()
torch.cuda.empty_cache()

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')

# --- Read train stems from label fraction file ---
with open('/content/drive/MyDrive/UNM_vertebras_seg_v3/label_fractions/frac_75/stems.txt') as f:
    train_stems = sorted([line.strip() for line in f if line.strip()])
print(f'Train stems (75%): {len(train_stems)}')

# --- Build val stems (always the same 44) ---
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'
assert not set(train_stems) & set(val_stems), 'Train/val overlap detected!'
print(f'Val stems: {len(val_stems)}')

# --- Overwrite splits_final.json ---
splits_final = [{"train": train_stems, "val": val_stems}]
splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump(splits_final, f, indent=2)
print(f'splits_final.json overwritten: {len(train_stems)} train + {len(val_stems)} val')

# --- Train ---
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_75pct')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 501 2d 0
# To resume: !nnUNetv2_train 501 2d 0 --c
elapsed = time.time() - t0
print(f'Training done in {elapsed/3600:.1f}h')

# --- Predict ---
imagesTs = os.path.join(os.environ['nnUNet_raw'], 'Dataset501_VFSS', 'imagesTs')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_75pct')
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 501 -c 2d -f 0

preds = [f for f in os.listdir(pred_dir) if f.endswith('.png')]
print(f'Predictions: {len(preds)} files in {pred_dir}')

# --- Evaluate ---
GT_DIR = os.path.join(BASE, 'test', 'masks')
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}

matched_stems = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
assert len(matched_stems) == len(gt_files), (
    f'Expected {len(gt_files)} matches, got {len(matched_stems)}'
)

per_image = []
for stem in matched_stems:
    gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
    pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))
    if pred.shape != gt.shape:
        pred = np.array(
            Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST)
        )
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

mean_f1 = float(np.mean([r['f1'] for r in per_image]))
mean_iou = float(np.mean([r['iou'] for r in per_image]))

# Save per-image CSV
csv_path = os.path.join(RESULTS_BASE, 'per_image_metrics_75pct.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
    writer.writeheader()
    writer.writerows(per_image)

# Save metrics JSON
metrics = {
    'fraction': 75,
    'num_train': len(train_stems),
    'num_val': len(val_stems),
    'mean_f1': mean_f1,
    'mean_iou': mean_iou,
    'num_test_images': len(per_image),
}
with open(os.path.join(RESULTS_BASE, 'metrics_75pct.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n75% labels ({len(train_stems)} train): F1={mean_f1:.4f}, IoU={mean_iou:.4f}')

In [ ]:
# === Cell D: Label efficiency summary (25% vs 50% vs 75% vs 100%) ===

import os, json, csv

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = os.path.join(BASE, 'nnunet_baseline_results')

rows = []

# Load fraction metrics
for frac in [25, 50, 75]:
    path = os.path.join(RESULTS_BASE, f'metrics_{frac}pct.json')
    with open(path) as f:
        m = json.load(f)
    rows.append({
        'fraction': frac,
        'num_train': m['num_train'],
        'mean_f1': m['mean_f1'],
        'mean_iou': m['mean_iou'],
    })

# Load 100% baseline (from the 3-run aggregated metrics)
path_100 = os.path.join(RESULTS_BASE, 'metrics.json')
with open(path_100) as f:
    m100 = json.load(f)
rows.append({
    'fraction': 100,
    'num_train': 218,
    'mean_f1': m100['mean_f1'],
    'mean_iou': m100['mean_iou'],
})

# Print table
print(f'{"Frac":>5s}  {"#Train":>6s}  {"F1":>7s}  {"IoU":>7s}')
print('-' * 30)
for r in rows:
    print(f'{r["fraction"]:>4d}%  {r["num_train"]:>6d}  {r["mean_f1"]:>7.4f}  {r["mean_iou"]:>7.4f}')

# Save CSV
csv_path = os.path.join(RESULTS_BASE, 'label_efficiency_summary.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['fraction', 'num_train', 'mean_f1', 'mean_iou'])
    writer.writeheader()
    writer.writerows(rows)
print(f'\nSaved: {csv_path}')

In [ ]:
# === Cell F: Generate pseudo-labels from nnU-Net 25% teacher ===

import gc
gc.collect()
import torch
torch.cuda.empty_cache()

import os, shutil, json
import numpy as np
from PIL import Image

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'

# --- Set teacher ---
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_25pct')
print(f'Teacher: {os.environ["nnUNet_results"]}')

# --- Prepare unlabeled pool in nnU-Net input format ---
UNLABELED_SRC = os.path.join(BASE, 'unlabeling_r10_max0', 'images')
TMP_INPUT = os.path.join(RESULTS_BASE, '_tmp_unlabeled_input_25')
TMP_OUTPUT = os.path.join(RESULTS_BASE, '_tmp_unlabeled_pred_25')

for d in [TMP_INPUT, TMP_OUTPUT]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

unlabeled_files = sorted([f for f in os.listdir(UNLABELED_SRC) if f.endswith('.png')])
for fname in unlabeled_files:
    stem = fname.replace('.png', '')
    Image.open(os.path.join(UNLABELED_SRC, fname)).convert('L').save(
        os.path.join(TMP_INPUT, f'{stem}_0000.png')
    )
print(f'Prepared {len(unlabeled_files)} unlabeled images in nnU-Net format')

# --- Run prediction with probabilities ---
!nnUNetv2_predict \
    -i {TMP_INPUT} \
    -o {TMP_OUTPUT} \
    -d 501 -c 2d -f 0 \
    --save_probabilities

# --- Filter by confidence and save pseudo-labels ---
PSEUDO_IMG_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_25pct_teacher', 'imagesTr')
PSEUDO_LBL_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_25pct_teacher', 'labelsTr')
for d in [PSEUDO_IMG_DIR, PSEUDO_LBL_DIR]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

CONF_THRESHOLD = 0.95
n_kept = 0
n_rejected = 0

npz_files = sorted([f for f in os.listdir(TMP_OUTPUT) if f.endswith('.npz')])
for npz_fname in npz_files:
    # Extract original stem (remove .npz)
    stem = npz_fname.replace('.npz', '')

    # Load probability map
    probs = np.load(os.path.join(TMP_OUTPUT, npz_fname))['probabilities']
    # probs shape: (n_classes, H, W) — use foreground channel
    if probs.ndim == 3 and probs.shape[0] == 2:
        fg_prob = probs[1]  # foreground = class 1
    elif probs.ndim == 3 and probs.shape[0] == 1:
        fg_prob = probs[0]
    else:
        fg_prob = probs

    fg_prob = np.asarray(fg_prob).squeeze()
    assert fg_prob.ndim == 2, f"Expected 2D foreground probability map, got shape {fg_prob.shape}"

    # Confidence: max(p, 1-p) per pixel, then mean
    confidence_map = np.maximum(fg_prob, 1.0 - fg_prob)
    mean_conf = confidence_map.mean()

    if mean_conf < CONF_THRESHOLD:
        n_rejected += 1
        continue

    # Binary pseudo-label
    mask = (fg_prob >= 0.5).astype(np.uint8)
    pseudo_stem = f'pseudo25_{stem}'

    # Save pseudo-label
    Image.fromarray(mask).save(os.path.join(PSEUDO_LBL_DIR, f'{pseudo_stem}.png'))

    # Copy corresponding input image
    src_img = os.path.join(TMP_INPUT, f'{stem}_0000.png')
    shutil.copy2(src_img, os.path.join(PSEUDO_IMG_DIR, f'{pseudo_stem}_0000.png'))

    n_kept += 1

print(f'\nPseudo-label generation (25% teacher):')
print(f'  Total unlabeled: {len(unlabeled_files)}')
print(f'  Kept (conf >= {CONF_THRESHOLD}): {n_kept}')
print(f'  Rejected: {n_rejected}')

# Cleanup temp folders
shutil.rmtree(TMP_INPUT)
shutil.rmtree(TMP_OUTPUT)
print('Temp folders cleaned up')


In [ ]:
# === Cell G: Create Dataset502_VFSS_SSL25, train, predict, evaluate ===

import gc
gc.collect()
import torch
torch.cuda.empty_cache()

import os, shutil, json, csv, re, time
import numpy as np
from PIL import Image

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'
RAW = os.environ['nnUNet_raw']

DS501 = os.path.join(RAW, 'Dataset501_VFSS')
DS502 = os.path.join(RAW, 'Dataset502_VFSS_SSL25')
PSEUDO_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_25pct_teacher')

# --- Create Dataset502 folders ---
for sub in ['imagesTr', 'labelsTr', 'imagesTs']:
    d = os.path.join(DS502, sub)
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

# --- Read stems ---
with open(os.path.join(BASE, 'label_fractions', 'frac_25', 'stems.txt')) as f:
    train_stems_25 = sorted([line.strip() for line in f if line.strip()])
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])
assert len(train_stems_25) == 66, f'Expected 66 train stems, got {len(train_stems_25)}'
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'

# --- Copy real cases from Dataset501 ---
real_stems = train_stems_25 + val_stems
for stem in real_stems:
    shutil.copy2(
        os.path.join(DS501, 'imagesTr', f'{stem}_0000.png'),
        os.path.join(DS502, 'imagesTr', f'{stem}_0000.png'),
    )
    shutil.copy2(
        os.path.join(DS501, 'labelsTr', f'{stem}.png'),
        os.path.join(DS502, 'labelsTr', f'{stem}.png'),
    )
print(f'Copied {len(real_stems)} real cases (66 train + 44 val) from Dataset501')

# --- Copy pseudo-labeled cases ---
pseudo_img_dir = os.path.join(PSEUDO_DIR, 'imagesTr')
pseudo_lbl_dir = os.path.join(PSEUDO_DIR, 'labelsTr')
pseudo_stems = sorted([
    f.replace('_0000.png', '')
    for f in os.listdir(pseudo_img_dir)
    if f.endswith('_0000.png')
])
for ps in pseudo_stems:
    shutil.copy2(
        os.path.join(pseudo_img_dir, f'{ps}_0000.png'),
        os.path.join(DS502, 'imagesTr', f'{ps}_0000.png'),
    )
    shutil.copy2(
        os.path.join(pseudo_lbl_dir, f'{ps}.png'),
        os.path.join(DS502, 'labelsTr', f'{ps}.png'),
    )
print(f'Copied {len(pseudo_stems)} pseudo-labeled cases')

# --- Copy imagesTs from Dataset501 ---
for fname in os.listdir(os.path.join(DS501, 'imagesTs')):
    shutil.copy2(
        os.path.join(DS501, 'imagesTs', fname),
        os.path.join(DS502, 'imagesTs', fname),
    )

# --- dataset.json ---
num_training = len(real_stems) + len(pseudo_stems)
ds_json = {
    'channel_names': {'0': 'Xray'},
    'labels': {'background': 0, 'vertebra': 1},
    'numTraining': num_training,
    'file_ending': '.png',
}
with open(os.path.join(DS502, 'dataset.json'), 'w') as f:
    json.dump(ds_json, f, indent=2)
print(f'dataset.json: numTraining={num_training}')

# --- Plan and preprocess ---
!nnUNetv2_plan_and_preprocess -d 502 --verify_dataset_integrity -c 2d

# --- Write splits_final.json AFTER preprocessing ---
split_train = sorted(train_stems_25 + pseudo_stems)
split_val = val_stems
assert not set(split_train) & set(split_val), 'Train/val overlap!'

# Verify all stems exist in Dataset502
ds502_imgs = set(f.replace('_0000.png', '') for f in os.listdir(os.path.join(DS502, 'imagesTr')))
for s in split_train + split_val:
    assert s in ds502_imgs, f'Stem {s} not found in Dataset502 imagesTr'

splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset502_VFSS_SSL25', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump([{'train': split_train, 'val': split_val}], f, indent=2)
print(f'splits_final.json: {len(split_train)} train, {len(split_val)} val')
print(f'Written to: {splits_path}')

# --- Train ---
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_ssl25_teacher')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 502 2d 0
# To resume: !nnUNetv2_train 502 2d 0 --c
print(f'Training done in {(time.time()-t0)/3600:.1f}h')

# --- Predict ---
imagesTs = os.path.join(DS502, 'imagesTs')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_ssl_25pct_teacher')
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 502 -c 2d -f 0

# --- Evaluate ---
GT_DIR = os.path.join(BASE, 'test', 'masks')
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}
matched = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
assert len(matched) == len(gt_files), f'Expected {len(gt_files)} matches, got {len(matched)}'

per_image = []
for stem in matched:
    gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
    pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))
    if pred.shape != gt.shape:
        pred = np.array(Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST))
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

mean_f1 = float(np.mean([r['f1'] for r in per_image]))
mean_iou = float(np.mean([r['iou'] for r in per_image]))

# Save CSV
csv_path = os.path.join(RESULTS_BASE, 'per_image_metrics_ssl_25pct_teacher.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
    writer.writeheader()
    writer.writerows(per_image)

# Save JSON
metrics = {
    'condition': 'nnunet_ssl_25pct_teacher',
    'real_train': 66,
    'pseudo_kept': len(pseudo_stems),
    'mean_f1': mean_f1,
    'mean_iou': mean_iou,
    'num_test_images': len(per_image),
}
with open(os.path.join(RESULTS_BASE, 'metrics_ssl_25pct_teacher.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nnnU-Net 25% supervised        = 0.896')
print(f'nnU-Net 25% + self-training   = {mean_f1:.4f}')
print(f'IoU: {mean_iou:.4f}')


In [ ]:
# === Cell H: Generate pseudo-labels from nnU-Net 100% teacher ===

import gc
gc.collect()
import torch
torch.cuda.empty_cache()

import os, shutil, json
import numpy as np
from PIL import Image

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'

# --- Set teacher ---
os.environ['nnUNet_results'] = os.path.join(WORKSPACE, 'nnUNet_results_run0')
print(f'Teacher: {os.environ["nnUNet_results"]}')

# --- Prepare unlabeled pool ---
UNLABELED_SRC = os.path.join(BASE, 'unlabeling_r10_max0', 'images')
TMP_INPUT = os.path.join(RESULTS_BASE, '_tmp_unlabeled_input_100')
TMP_OUTPUT = os.path.join(RESULTS_BASE, '_tmp_unlabeled_pred_100')

for d in [TMP_INPUT, TMP_OUTPUT]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

unlabeled_files = sorted([f for f in os.listdir(UNLABELED_SRC) if f.endswith('.png')])
for fname in unlabeled_files:
    stem = fname.replace('.png', '')
    Image.open(os.path.join(UNLABELED_SRC, fname)).convert('L').save(
        os.path.join(TMP_INPUT, f'{stem}_0000.png')
    )
print(f'Prepared {len(unlabeled_files)} unlabeled images in nnU-Net format')

# --- Run prediction with probabilities ---
!nnUNetv2_predict \
    -i {TMP_INPUT} \
    -o {TMP_OUTPUT} \
    -d 501 -c 2d -f 0 \
    --save_probabilities

# --- Filter by confidence and save pseudo-labels ---
PSEUDO_IMG_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_100pct_teacher', 'imagesTr')
PSEUDO_LBL_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_100pct_teacher', 'labelsTr')
for d in [PSEUDO_IMG_DIR, PSEUDO_LBL_DIR]:
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

CONF_THRESHOLD = 0.95
n_kept = 0
n_rejected = 0

npz_files = sorted([f for f in os.listdir(TMP_OUTPUT) if f.endswith('.npz')])
for npz_fname in npz_files:
    stem = npz_fname.replace('.npz', '')

    probs = np.load(os.path.join(TMP_OUTPUT, npz_fname))['probabilities']
    if probs.ndim == 3 and probs.shape[0] == 2:
        fg_prob = probs[1]
    elif probs.ndim == 3 and probs.shape[0] == 1:
        fg_prob = probs[0]
    else:
        fg_prob = probs

    fg_prob = np.asarray(fg_prob).squeeze()
    assert fg_prob.ndim == 2, f"Expected 2D foreground probability map, got shape {fg_prob.shape}"

    confidence_map = np.maximum(fg_prob, 1.0 - fg_prob)
    mean_conf = confidence_map.mean()

    if mean_conf < CONF_THRESHOLD:
        n_rejected += 1
        continue

    mask = (fg_prob >= 0.5).astype(np.uint8)
    pseudo_stem = f'pseudo100_{stem}'

    Image.fromarray(mask).save(os.path.join(PSEUDO_LBL_DIR, f'{pseudo_stem}.png'))
    src_img = os.path.join(TMP_INPUT, f'{stem}_0000.png')
    shutil.copy2(src_img, os.path.join(PSEUDO_IMG_DIR, f'{pseudo_stem}_0000.png'))

    n_kept += 1

print(f'\nPseudo-label generation (100% teacher):')
print(f'  Total unlabeled: {len(unlabeled_files)}')
print(f'  Kept (conf >= {CONF_THRESHOLD}): {n_kept}')
print(f'  Rejected: {n_rejected}')

shutil.rmtree(TMP_INPUT)
shutil.rmtree(TMP_OUTPUT)
print('Temp folders cleaned up')


In [ ]:
# === Cell I: Create Dataset503_VFSS_SSL100, train, predict, evaluate ===

import gc
gc.collect()
import torch
torch.cuda.empty_cache()

import os, shutil, json, csv, re, time
import numpy as np
from PIL import Image

WORKSPACE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_workspace'
BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'
RESULTS_BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'
RAW = os.environ['nnUNet_raw']

DS501 = os.path.join(RAW, 'Dataset501_VFSS')
DS503 = os.path.join(RAW, 'Dataset503_VFSS_SSL100')
PSEUDO_DIR = os.path.join(RESULTS_BASE, 'pseudo_labels_100pct_teacher')

# --- Create Dataset503 folders ---
for sub in ['imagesTr', 'labelsTr', 'imagesTs']:
    d = os.path.join(DS503, sub)
    if os.path.isdir(d):
        shutil.rmtree(d)
    os.makedirs(d)

# --- Read stems ---
train_stems_100 = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'train', 'images'))
    if f.endswith('.png')
])
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])
assert len(train_stems_100) == 218, f'Expected 218 train stems, got {len(train_stems_100)}'
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'

# --- Copy real cases from Dataset501 ---
real_stems = train_stems_100 + val_stems
for stem in real_stems:
    shutil.copy2(
        os.path.join(DS501, 'imagesTr', f'{stem}_0000.png'),
        os.path.join(DS503, 'imagesTr', f'{stem}_0000.png'),
    )
    shutil.copy2(
        os.path.join(DS501, 'labelsTr', f'{stem}.png'),
        os.path.join(DS503, 'labelsTr', f'{stem}.png'),
    )
print(f'Copied {len(real_stems)} real cases (218 train + 44 val) from Dataset501')

# --- Copy pseudo-labeled cases ---
pseudo_img_dir = os.path.join(PSEUDO_DIR, 'imagesTr')
pseudo_lbl_dir = os.path.join(PSEUDO_DIR, 'labelsTr')
pseudo_stems = sorted([
    f.replace('_0000.png', '')
    for f in os.listdir(pseudo_img_dir)
    if f.endswith('_0000.png')
])
for ps in pseudo_stems:
    shutil.copy2(
        os.path.join(pseudo_img_dir, f'{ps}_0000.png'),
        os.path.join(DS503, 'imagesTr', f'{ps}_0000.png'),
    )
    shutil.copy2(
        os.path.join(pseudo_lbl_dir, f'{ps}.png'),
        os.path.join(DS503, 'labelsTr', f'{ps}.png'),
    )
print(f'Copied {len(pseudo_stems)} pseudo-labeled cases')

# --- Copy imagesTs ---
for fname in os.listdir(os.path.join(DS501, 'imagesTs')):
    shutil.copy2(
        os.path.join(DS501, 'imagesTs', fname),
        os.path.join(DS503, 'imagesTs', fname),
    )

# --- dataset.json ---
num_training = len(real_stems) + len(pseudo_stems)
ds_json = {
    'channel_names': {'0': 'Xray'},
    'labels': {'background': 0, 'vertebra': 1},
    'numTraining': num_training,
    'file_ending': '.png',
}
with open(os.path.join(DS503, 'dataset.json'), 'w') as f:
    json.dump(ds_json, f, indent=2)
print(f'dataset.json: numTraining={num_training}')

# --- Plan and preprocess ---
!nnUNetv2_plan_and_preprocess -d 503 --verify_dataset_integrity -c 2d

# --- Write splits_final.json AFTER preprocessing ---
split_train = sorted(train_stems_100 + pseudo_stems)
split_val = val_stems
assert not set(split_train) & set(split_val), 'Train/val overlap!'

ds503_imgs = set(f.replace('_0000.png', '') for f in os.listdir(os.path.join(DS503, 'imagesTr')))
for s in split_train + split_val:
    assert s in ds503_imgs, f'Stem {s} not found in Dataset503 imagesTr'

splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset503_VFSS_SSL100', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump([{'train': split_train, 'val': split_val}], f, indent=2)
print(f'splits_final.json: {len(split_train)} train, {len(split_val)} val')
print(f'Written to: {splits_path}')

# --- Train ---
results_dir = os.path.join(WORKSPACE, 'nnUNet_results_ssl100_teacher')
os.makedirs(results_dir, exist_ok=True)
os.environ['nnUNet_results'] = results_dir
print(f'nnUNet_results -> {results_dir}')

t0 = time.time()
!nnUNetv2_train 503 2d 0
# To resume: !nnUNetv2_train 503 2d 0 --c
print(f'Training done in {(time.time()-t0)/3600:.1f}h')

# --- Predict ---
imagesTs = os.path.join(DS503, 'imagesTs')
pred_dir = os.path.join(RESULTS_BASE, 'predictions_ssl_100pct_teacher')
if os.path.isdir(pred_dir):
    shutil.rmtree(pred_dir)
os.makedirs(pred_dir)

!nnUNetv2_predict \
    -i {imagesTs} \
    -o {pred_dir} \
    -d 503 -c 2d -f 0

# --- Evaluate ---
GT_DIR = os.path.join(BASE, 'test', 'masks')
gt_files = sorted([f for f in os.listdir(GT_DIR) if f.endswith('.png')])
gt_by_stem = {f.replace('.png', ''): os.path.join(GT_DIR, f) for f in gt_files}


def normalize_pred_stem(fname):
    stem = fname.replace('.png', '')
    stem = re.sub(r'_\d{4}$', '', stem)
    return stem


def compute_dice_iou(pred_bin, gt_bin):
    intersection = np.sum(pred_bin * gt_bin)
    sum_pred = np.sum(pred_bin)
    sum_gt = np.sum(gt_bin)
    dice = (2.0 * intersection) / (sum_pred + sum_gt) if (sum_pred + sum_gt) > 0 else 1.0
    union = sum_pred + sum_gt - intersection
    iou = intersection / union if union > 0 else 1.0
    return dice, iou


pred_files = sorted([f for f in os.listdir(pred_dir) if f.endswith('.png')])
pred_by_stem = {normalize_pred_stem(f): os.path.join(pred_dir, f) for f in pred_files}
matched = sorted(set(pred_by_stem.keys()) & set(gt_by_stem.keys()))
assert len(matched) == len(gt_files), f'Expected {len(gt_files)} matches, got {len(matched)}'

per_image = []
for stem in matched:
    gt = np.array(Image.open(gt_by_stem[stem]).convert('L'))
    pred = np.array(Image.open(pred_by_stem[stem]).convert('L'))
    if pred.shape != gt.shape:
        pred = np.array(Image.fromarray(pred).resize((gt.shape[1], gt.shape[0]), Image.NEAREST))
    gt_bin = (gt > 0).astype(np.float32)
    pred_bin = (pred > 0).astype(np.float32)
    dice, iou = compute_dice_iou(pred_bin, gt_bin)
    per_image.append({'stem': stem, 'f1': dice, 'iou': iou})

mean_f1 = float(np.mean([r['f1'] for r in per_image]))
mean_iou = float(np.mean([r['iou'] for r in per_image]))

csv_path = os.path.join(RESULTS_BASE, 'per_image_metrics_ssl_100pct_teacher.csv')
with open(csv_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['stem', 'f1', 'iou'])
    writer.writeheader()
    writer.writerows(per_image)

metrics = {
    'condition': 'nnunet_ssl_100pct_teacher',
    'real_train': 218,
    'pseudo_kept': len(pseudo_stems),
    'mean_f1': mean_f1,
    'mean_iou': mean_iou,
    'num_test_images': len(per_image),
}
with open(os.path.join(RESULTS_BASE, 'metrics_ssl_100pct_teacher.json'), 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nnnU-Net 100% supervised         = 0.908')
print(f'nnU-Net 100% + self-training    = {mean_f1:.4f}')
print(f'IoU: {mean_iou:.4f}')


In [ ]:
# === Cell J: Self-training summary table ===

import os, json

RESULTS_BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3/nnunet_baseline_results'

# Load metrics
with open(os.path.join(RESULTS_BASE, 'metrics_ssl_25pct_teacher.json')) as f:
    m25 = json.load(f)
with open(os.path.join(RESULTS_BASE, 'metrics_ssl_100pct_teacher.json')) as f:
    m100 = json.load(f)

rows = [
    ('nnU-Net 25% supervised',      66,  0,                  0.896),
    ('nnU-Net 25% + self-training',  66,  m25['pseudo_kept'], m25['mean_f1']),
    ('nnU-Net 100% supervised',      218, 0,                  0.908),
    ('nnU-Net 100% + self-training', 218, m100['pseudo_kept'], m100['mean_f1']),
]

print(f'{"Condition":<35s}  {"Real":>5s}  {"Pseudo":>6s}  {"F1":>7s}')
print('-' * 60)
for name, real, pseudo, f1 in rows:
    print(f'{name:<35s}  {real:>5d}  {pseudo:>6d}  {f1:>7.4f}')

print(f'\nDataset502 total training: {66 + 44 + m25["pseudo_kept"]}')
print(f'Dataset503 total training: {218 + 44 + m100["pseudo_kept"]}')


In [ ]:
# === Cell E: Restore 100% split and disconnect runtime ===

import os, json

BASE = '/content/drive/MyDrive/UNM_vertebras_seg_v3'

# Rebuild the full 100% train/val split
train_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'train', 'images'))
    if f.endswith('.png')
])
val_stems = sorted([
    f.replace('.png', '')
    for f in os.listdir(os.path.join(BASE, 'val', 'images'))
    if f.endswith('.png')
])

assert len(train_stems) == 218, f'Expected 218 train stems, got {len(train_stems)}'
assert len(val_stems) == 44, f'Expected 44 val stems, got {len(val_stems)}'

# Restore splits_final.json to 100%
splits_final = [{"train": train_stems, "val": val_stems}]
splits_path = os.path.join(
    os.environ['nnUNet_preprocessed'], 'Dataset501_VFSS', 'splits_final.json'
)
with open(splits_path, 'w') as f:
    json.dump(splits_final, f, indent=2)

print(f'Restored splits_final.json to 100%: {len(train_stems)} train + {len(val_stems)} val')
print(f'Path: {splits_path}')

from google.colab import runtime
runtime.unassign()